In [1]:
import duckdb
import time

def extract_overture_india():
    print("Initializing DuckDB spatial engine...")
    con = duckdb.connect("overture_spatial.db")
    con.execute("INSTALL spatial; LOAD spatial;")
    con.execute("INSTALL httpfs; LOAD httpfs;")
    
    # Base S3 Path for Overture (August 2026 Stable Release)
    S3_BASE = "s3://overturemaps-us-west-2/release/2026-08-19.0"
    
    # Bounding Box for India
    INDIA_BBOX = "bbox.xmin >= 68.1 AND bbox.xmax <= 97.4 AND bbox.ymin >= 6.5 AND bbox.ymax <= 37.1"

    # --- 4. BASE (Water & Parks for Physical Barriers) ---
    print("Streaming Environmental Boundaries...")
    con.execute(f"""
        COPY (
            SELECT 
                id, 
                subtype, 
                geometry
            FROM read_parquet(
                '{S3_BASE}/theme=base/type=*/*', 
                filename=true, 
                hive_partitioning=1,
                union_by_name=true
            )
            WHERE {INDIA_BBOX} 
              AND subtype IN ('park', 'water', 'lake', 'river', 'forest', 'industrial')
        ) TO 'india_base.parquet' (FORMAT PARQUET, COMPRESSION ZSTD);
    """)

extract_overture_india()

Initializing DuckDB spatial engine...
Streaming Environmental Boundaries...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))